# HR Policy RAG — Colab notebook (historical)

> **Portfolio note:** The maintained, modular pipeline lives in `../src/` and `../scripts/`.
> Do **not** use the Nestlé PDF from bootcamp — use `../data/sample/employee_handbook_sample.txt` or your own `data/raw/` files.
> Clear outputs before committing (`jupyter nbconvert --clear-output`).


In [ ]:
!pip install transformers langchain chromadb sentence-transformers pypdf gradio


In [ ]:
import transformers
import langchain
import chromadb
import gradio

print("All good ✅")

In [ ]:
!pip install langchain-community

In [ ]:
import langchain_community
print("ok")

In [ ]:
# Colab: use sample handbook or upload your own PDF
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from pathlib import Path

# Example for local repo: TextLoader("../data/sample/employee_handbook_sample.txt")
loader = PyPDFLoader("the_nestle_hr_policy_pdf_2012.pdf")  # replace in Colab upload
documents = loader.load()

print(len(documents))
print(documents[0].page_content[:300])


In [ ]:
#Split Text
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # size of each piece
    chunk_overlap=50     # small overlap for context
)

docs = text_splitter.split_documents(documents)

print(len(docs))

In [ ]:
print(len(docs))

In [ ]:
# Embeddings
# Convert text into numerical vectors so AI can understand meaning
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
# Store in Vector Database
# Store embeddings in ChromaDB so we can search them efficiently

from langchain_community.vectorstores import Chroma

db = Chroma.from_documents(docs, embeddings)

# Create retriever (this will fetch relevant chunks)
retriever = db.as_retriever()

print("Vector database created successfully ✅")

In [ ]:
# Test
query = "employee benefits"

results = retriever.invoke(query)

print(results[0].page_content)

In [ ]:

# Build QA System (RAG)
# Combine retriever + model to generate answers

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200
)


def answer_question(question):
    # Step 1: Retrieve relevant chunks
    docs = retriever.invoke(question)

    # Step 2: Combine context
    context = "\n".join([doc.page_content for doc in docs])

    # Step 3: Prompt
    prompt = f"""
You are an HR assistant for Nestlé.

Answer the question based ONLY on the context below.

Context:
{context}

Question:
{question}

Answer:
"""

    # Step 4: Generate answer
    result = pipe(prompt)[0]["generated_text"]

    return result

In [ ]:
print(answer_question("What are employee benefits?"))